# Newton Solver Convergence

This notebook covers Newton-Raphson solver configuration and DC
convergence aids in Pulsim (v2 API).

> **v2 note:** v1 exposed a large set of solver-introspection objects
> (`NewtonOptions`, `Tolerances`, `ConvergenceHistory`,
> `IterationRecord`, `GminConfig`, `SourceSteppingConfig`,
> `DCConvergenceConfig`, `VariableConvergence`, `PerVariableConvergence`,
> `SolverStatus`, `BenchmarkResult`). These internal classes are **not
> part of v2** (a documented gap, see `migration-from-v1.md` §7). v2
> replaces them with a much smaller surface:
>
> - Newton controls are keyword args to `p.simulate(...)`:
>   `max_newton_iterations`, `tol_newton_res`, `tol_newton_dx`,
>   `enable_newton_line_search`, `enable_newton_lm` (also bundled in
>   `p.SolverOptions`).
> - DC convergence uses `p.compute_dc_op(b, strategy=...)` with a
>   strategy ladder (`'auto'`, `'naive'`, `'source_step'`,
>   `'gmin_step'`, `'pseudo_trans'`, `'settle'`) plus `p.SourceStepConfig` /
>   `p.SettleConfig` (alias `p.PseudoTransientConfig`).

## Contents
1. Newton Solver Options (v2)
2. Tolerances Configuration
3. DC Operating Point + verbose diagnostics
4. DC Convergence Aids (strategy ladder)
5. Troubleshooting Convergence Issues

In [1]:
import pulsim as p
import numpy as np

## 1. Newton Solver Options (v2)

In v2 the Newton-Raphson controls are keyword arguments to
`p.simulate(...)` (and `p.compute_dc_op` uses sensible defaults):

| v1 `NewtonOptions` field | v2 equivalent |
|--------------------------|---------------|
| `max_iterations`         | `simulate(..., max_newton_iterations=)` |
| residual tolerance       | `simulate(..., tol_newton_res=)` |
| step (Δx) tolerance      | `simulate(..., tol_newton_dx=)` |
| damping / line search    | `simulate(..., enable_newton_line_search=True)` |
| Levenberg-Marquardt      | `simulate(..., enable_newton_lm=True)` |

These can also be bundled into a `p.SolverOptions` object and passed
via `simulate(..., solver=opts)`.

In [2]:
# Inspect the v2 SolverOptions bundle (defaults)
opts = p.SolverOptions()
print("Default SolverOptions (Newton-relevant fields):")
print(f"  max_newton_iterations:      {opts.max_newton_iterations}")
print(f"  tol_newton_res:             {opts.tol_newton_res}")
print(f"  tol_newton_dx:              {opts.tol_newton_dx}")
print(f"  enable_newton_line_search:  {opts.enable_newton_line_search}")
print(f"  enable_newton_lm:           {opts.enable_newton_lm}")

# Customize (e.g. allow more iterations + tighter residual)
opts.max_newton_iterations = 100
opts.tol_newton_res = 1e-12
print("\nCustomized:")
print(f"  max_newton_iterations: {opts.max_newton_iterations}")
print(f"  tol_newton_res:        {opts.tol_newton_res}")

Default SolverOptions (Newton-relevant fields):
  max_newton_iterations:      0
  tol_newton_res:             None
  tol_newton_dx:              None
  enable_newton_line_search:  None
  enable_newton_lm:           None

Customized:
  max_newton_iterations: 100
  tol_newton_res:        1e-12


## 2. Tolerances Configuration

v1 had a dedicated `Tolerances` object (vntol / iatol / reltol …).
In v2 the convergence criteria are part of the solver controls:

- **`tol_newton_res`** — residual norm tolerance (when `‖F(x)‖` is
  below this, the Newton iteration has converged).
- **`tol_newton_dx`** — step norm tolerance (when the update `‖Δx‖`
  is below this, iteration stops).
- **`atol` / `rtol`** — absolute / relative LTE tolerances used by the
  adaptive (`engine='dsed'`) integrator for time-step control.

A `None` field means "use the engine's built-in default".

In [3]:
# Default LTE tolerances (used by the adaptive dsed integrator)
opts = p.SolverOptions()
print("Default SolverOptions tolerances:")
print(f"  atol (absolute LTE): {opts.atol}")
print(f"  rtol (relative LTE): {opts.rtol}")
print(f"  tol_newton_res:      {opts.tol_newton_res}  (None → engine default)")
print(f"  tol_newton_dx:       {opts.tol_newton_dx}  (None → engine default)")

# Tighter tolerances for precision
opts_tight = p.SolverOptions()
opts_tight.atol = 1e-12
opts_tight.rtol = 1e-9
opts_tight.tol_newton_res = 1e-12

print("\nTight tolerances:")
print(f"  atol:           {opts_tight.atol}")
print(f"  rtol:           {opts_tight.rtol}")
print(f"  tol_newton_res: {opts_tight.tol_newton_res}")

Default SolverOptions tolerances:
  atol (absolute LTE): None
  rtol (relative LTE): None
  tol_newton_res:      None  (None → engine default)
  tol_newton_dx:       None  (None → engine default)

Tight tolerances:
  atol:           1e-12
  rtol:           1e-09
  tol_newton_res: 1e-12


## 3. DC Operating Point & Diagnostics

v1 returned a rich `dc_result` object with a `newton_result`
(`.iterations`, `.final_residual`, `.status`) and a
`ConvergenceHistory`. In v2, `p.compute_dc_op(b)` returns the
**solution vector** directly (a NumPy array indexed by node id), and a
`verbose=True` flag prints the strategy ladder + convergence
diagnostics to stdout.

In [4]:
# Create a circuit for convergence analysis (resistive divider + cap)
def build_test_circuit():
    b = p.CircuitBuilder()
    b.add_voltage_source("V1", "n1", "gnd", 10.0)
    b.add_resistor("R1", "n1", "n2", 1000.0)
    b.add_resistor("R2", "n2", "n3", 2000.0)
    b.add_resistor("R3", "n3", "gnd", 3000.0)
    b.add_capacitor("C1", "n2", "gnd", 1e-6)
    return b

b = build_test_circuit()

# DC operating point — returns the solution vector
dc = p.compute_dc_op(b)

print("DC Operating Point (v2 returns a NumPy solution vector):")
print(f"  V(n1) = {dc[b.node_id_of('n1')]:.4f} V")
print(f"  V(n2) = {dc[b.node_id_of('n2')]:.4f} V")
print(f"  V(n3) = {dc[b.node_id_of('n3')]:.4f} V")

# Cross-check against the resistive divider (capacitor = open at DC):
#   V(n2) = 10 * (R2 + R3) / (R1 + R2 + R3) = 10 * 5000/6000
print(f"\nExpected V(n2) = 10 * 5000/6000 = {10 * 5000/6000:.4f} V")

DC Operating Point (v2 returns a NumPy solution vector):
  V(n1) = 10.0000 V
  V(n2) = 8.3333 V
  V(n3) = 5.0000 V

Expected V(n2) = 10 * 5000/6000 = 8.3333 V


In [5]:
# Verbose DC solve — prints the strategy ladder + convergence diagnostics.
# (v2 has no ConvergenceHistory / IterationRecord objects to introspect;
#  the diagnostics are emitted as a log instead.)
print("Verbose DC operating-point solve:")
print("-" * 60)
dc = p.compute_dc_op(b, strategy="auto", verbose=True)
print("-" * 60)
print(f"Solved: V(n2) = {dc[b.node_id_of('n2')]:.4f} V")

Verbose DC operating-point solve:
------------------------------------------------------------
  compute_dc_op[auto]: DC operating point solved by naive (1 rung(s), residual 1.000e-11)
------------------------------------------------------------
Solved: V(n2) = 8.3333 V


## 4. DC Convergence Aids (strategy ladder)

For difficult circuits, v2 selects a **DC strategy** via the
`strategy=` argument to `p.compute_dc_op`:

- **`'auto'`** (default) — tries the cheap direct solve, then falls
  back to source-stepping / pseudo-transient if needed.
- **`'naive'`** — single direct Newton solve (fastest, for easy
  circuits).
- **`'source_step'`** — gradually ramps the sources from 0 to full
  value (`p.SourceStepConfig`).
- **`'pseudo_trans'`** — the kernel's pseudo-transient continuation
  rung (its own defaults; it takes no config): every node gets a
  1/dt shunt to ground that fades as dt grows. (Until 2026-11 the
  shunt was also put on the voltage-source rows, and it diverged on
  the divider below, ||F|| ~ 1e84.)
- **`'settle'`** — runs a real transient until it settles to steady
  state (`p.SettleConfig`, alias `p.PseudoTransientConfig`). Since v2.0
  passing a config with `strategy='pseudo_trans'` raises.

A small node-to-ground conductance floor (`gmin`) is applied
automatically, so v1's explicit `GminConfig` is no longer needed.

In [6]:
# Strategy: 'naive' — a single direct Newton solve (fastest path).
# An automatic gmin floor keeps the MNA matrix non-singular, so the
# explicit GminConfig from v1 is no longer required.
dc_naive = p.compute_dc_op(b, strategy="naive")
print("strategy='naive' (direct solve):")
print(f"  V(n2) = {dc_naive[b.node_id_of('n2')]:.4f} V")

strategy='naive' (direct solve):
  V(n2) = 8.3333 V


In [7]:
# Strategy: 'source_step' — ramp the sources from 0 to full value.
src_cfg = p.SourceStepConfig()
print("SourceStepConfig (defaults):")
print(f"  n_steps:     {src_cfg.n_steps}")
print(f"  dt:          {src_cfg.dt}")
print(f"  t_per_step:  {src_cfg.t_per_step}")

# Use more steps for a gentler ramp on a hard circuit
src_cfg.n_steps = 20
dc_src = p.compute_dc_op(b, strategy="source_step", source_step=src_cfg)
print(f"\nstrategy='source_step' (n_steps={src_cfg.n_steps}):")
print(f"  V(n2) = {dc_src[b.node_id_of('n2')]:.4f} V")

SourceStepConfig (defaults):
  n_steps:     10
  dt:          1e-06
  t_per_step:  0.0005

strategy='source_step' (n_steps=20):
  V(n2) = 8.3333 V


In [8]:
# Strategy: 'settle' — run a transient until it settles (SettleConfig).
# ('pseudo_trans' is the kernel's continuation rung and takes no config.)
ptran_cfg = p.SettleConfig()
print("SettleConfig (defaults):")
print(f"  dt:          {ptran_cfg.dt}")
print(f"  t_check:     {ptran_cfg.t_check}")
print(f"  t_settle:    {ptran_cfg.t_settle}")
print(f"  tol_steady:  {ptran_cfg.tol_steady}")

# This RC node has tau = R||... * C ~ a few ms, so allow enough settle time
ptran_cfg.dt = 1e-5
ptran_cfg.t_check = 5e-3
ptran_cfg.t_settle = 50e-3
ptran_cfg.tol_steady = 1e-3

dc_ptran = p.compute_dc_op(b, strategy="settle", settle=ptran_cfg)
print(f"\nstrategy='settle' (t_settle={ptran_cfg.t_settle*1e3:.0f} ms):")
print(f"  V(n2) = {dc_ptran[b.node_id_of('n2')]:.4f} V")

SettleConfig (defaults):
  dt:          1e-06
  t_check:     0.0001
  t_settle:    0.005
  tol_steady:  0.001

strategy='settle' (t_settle=50 ms):
  V(n2) = 8.3333 V


In [9]:
# v1 bundled everything in a DCConvergenceConfig object; v2 has no such
# umbrella class (a documented gap). Instead you pick a strategy and pass
# the matching config. All five converge to the same answer here:
print("DC strategy comparison (same circuit, V(n2) target = 8.3333 V):\n")

results = {
    "auto":         p.compute_dc_op(b, strategy="auto"),
    "naive":        p.compute_dc_op(b, strategy="naive"),
    "source_step":  p.compute_dc_op(b, strategy="source_step", source_step=src_cfg),
    "pseudo_trans": p.compute_dc_op(b, strategy="pseudo_trans"),
    "settle":       p.compute_dc_op(b, strategy="settle", settle=ptran_cfg),
}
for name, sol in results.items():
    print(f"  {name:13s}: V(n2) = {sol[b.node_id_of('n2')]:.4f} V")

DC strategy comparison (same circuit, V(n2) target = 8.3333 V):

  auto         : V(n2) = 8.3333 V
  naive        : V(n2) = 8.3333 V
  source_step  : V(n2) = 8.3333 V
  pseudo_trans : V(n2) = 8.3333 V
  settle       : V(n2) = 8.3333 V


## 5. Troubleshooting Convergence Issues

Common convergence problems and solutions:

In [10]:
# Practical example: a nonlinear diode circuit needs a real Newton solve.
# v2 has no per-variable convergence introspection (VariableConvergence /
# PerVariableConvergence are v1-only); instead you rely on the strategy
# ladder and tune the Newton kwargs / solver when a circuit is stiff.

# One Shockley diode in a resistive divider — a small nonlinear DC problem.
b_nl = p.CircuitBuilder()
b_nl.add_voltage_source("V1", "vin", "gnd", 5.0)
b_nl.add_resistor("R1", "vin", "mid", 1000.0)
b_nl.add_shockley_diode("D1", "mid", "out", I_S=1e-12, n=1.0)
b_nl.add_resistor("R2", "out", "gnd", 1000.0)

# Source-stepping is a robust strategy for nonlinear DC problems.
dc_nl = p.compute_dc_op(b_nl, strategy="source_step")
v_mid = dc_nl[b_nl.node_id_of("mid")]
v_out = dc_nl[b_nl.node_id_of("out")]
print("Nonlinear diode divider DC operating point:")
print(f"  V(mid) = {v_mid:.4f} V")
print(f"  V(out) = {v_out:.4f} V")
print(f"  Diode drop V(mid) - V(out) = {v_mid - v_out:.4f} V  (≈ forward drop)")

Nonlinear diode divider DC operating point:
  V(mid) = 2.7782 V
  V(out) = 2.2218 V
  Diode drop V(mid) - V(out) = 0.5564 V  (≈ forward drop)


### Convergence Troubleshooting Guide

| Problem | Symptom | Solution |
|---------|---------|----------|
| Stiff circuit | Slow convergence | Reduce timestep, use tighter tolerances |
| Floating nodes | NaN values | Check all nodes have DC path to ground |
| Large dynamic range | Oscillating residual | Enable Gmin stepping |
| Nonlinear devices | No convergence | Enable source stepping |
| Startup issues | First iteration fails | Use pseudo-transient continuation |

In [11]:
# v1 exposed a SolverStatus enum (Success / Diverging / SingularMatrix /
# MaxIterationsReached / ...). v2 raises on hard failure and otherwise
# returns a result, so the practical check is "did simulate() complete
# and produce finite states?".
#
# Here we run a transient and tighten the Newton controls via direct
# simulate(...) kwargs (the warning-free way to set them on the PWL engine).
res = p.simulate(
    b,
    t_end=5e-3,
    dt=1e-4,
    max_newton_iterations=100,
    tol_newton_res=1e-12,
    tol_newton_dx=1e-12,
    enable_newton_line_search=True,
)

states = np.array(res.states)
converged = np.all(np.isfinite(states))
print(f"Transient completed: {res.num_steps()} steps")
print(f"  All states finite (converged): {converged}")
print(f"  Final V(n2) = {states[-1, b.node_id_of('n2')]:.4f} V")

Transient completed: 51 steps
  All states finite (converged): True
  Final V(n2) = 8.3115 V


## Summary

### v2 solver-control surface

| v1 class (removed) | v2 replacement |
|--------------------|----------------|
| `NewtonOptions` | `simulate(..., max_newton_iterations=, tol_newton_res=, tol_newton_dx=, enable_newton_line_search=, enable_newton_lm=)` or `p.SolverOptions` |
| `Tolerances` | `tol_newton_res`, `tol_newton_dx` (Newton); `atol`, `rtol` (adaptive LTE) |
| `DCConvergenceConfig` | `p.compute_dc_op(b, strategy=...)` |
| `GminConfig` | automatic gmin floor (no config needed) |
| `SourceSteppingConfig` | `p.SourceStepConfig` + `strategy='source_step'` |
| `PseudoTransientConfig` | `p.SettleConfig` + `strategy='settle'` (`'pseudo_trans'` is the kernel rung, no config) |
| `ConvergenceHistory` / `IterationRecord` | `compute_dc_op(..., verbose=True)` log |
| `VariableConvergence` / `SolverStatus` | result is finite ⇒ converged; hard failure raises |

### Example Usage

```python
import pulsim as p

# DC operating point with a chosen strategy
dc = p.compute_dc_op(b, strategy="source_step",
                     source_step=p.SourceStepConfig())      # returns a NumPy vector
v_n2 = dc[b.node_id_of("n2")]

# Pseudo-transient continuation for a stubborn circuit
pt = p.SettleConfig(); pt.t_settle = 50e-3
dc = p.compute_dc_op(b, strategy="settle", settle=pt)

# Transient with tightened Newton controls
res = p.simulate(b, t_end=..., dt=...,
                 max_newton_iterations=100,
                 tol_newton_res=1e-12,
                 enable_newton_line_search=True)
assert np.all(np.isfinite(np.array(res.states)))  # converged
```

> v2 gaps used in this notebook: the v1 solver-introspection classes
> (`NewtonOptions`, `Tolerances`, `ConvergenceHistory`,
> `IterationRecord`, `GminConfig`, `SourceSteppingConfig`,
> `DCConvergenceConfig`, `VariableConvergence`,
> `PerVariableConvergence`, `SolverStatus`, `BenchmarkResult`) are not
> present in v2 (documented gap, `migration-from-v1.md` §7).

**Next:** [Validation Framework](08_validation.ipynb)